# GPGPU Tasks 2026 — Colab Runner (CUDA)

**Runtime → Change runtime type → T4 GPU.**

Workflow: правишь код локально → `git push` в ветку `task03` → здесь ячейка 4 (pull + make + запуск).


In [ ]:
# 1. GPU
!nvidia-smi

In [ ]:
# 2. [1 раз за сессию] Клонирование + зависимости
BRANCH = "task03"
%cd /content
!rm -rf /content/GPGPUTasks2026
!git clone -b $BRANCH https://github.com/BOGOMOLOV-ARSENIQ/GPGPUTasks2026.git /content/GPGPUTasks2026
!sudo apt-get update -qq && sudo apt-get install -yqq unzip libgtest-dev libgraphicsmagick++1-dev

# Colab на Ubuntu 24.04: пакета libspirv-reflect-dev там нет (он есть только с 26.04),
# а проекту он не нужен (spirv_reflect лежит в libs/gpu) - выкидываем его из apt install скрипта
!sed -i 's/ libspirv-reflect-dev//' /content/GPGPUTasks2026/scripts/linux/install_vulkan_sdk.sh

# Ставим Vulkan SDK в чистой папке, полный лог - в /content/vulkan_install.log
!rm -rf /tmp/vk && mkdir -p /tmp/vk && cd /tmp/vk && bash /content/GPGPUTasks2026/scripts/linux/install_vulkan_sdk.sh --no-validation-layers > /content/vulkan_install.log 2>&1 || (tail -40 /content/vulkan_install.log; echo "VULKAN INSTALL FAILED")

import os
assert os.path.exists("/usr/local/include/vulkan/vulkan.h"), "Vulkan не установился - смотри лог выше"
print("Vulkan SDK установлен")


In [ ]:
# 3. [1 раз за сессию] Конфигурация CMake (75 = T4)
!mkdir -p /content/GPGPUTasks2026/build
%cd /content/GPGPUTasks2026/build
!cmake -DCMAKE_BUILD_TYPE=Release -DGPU_CUDA_SUPPORT=ON -DCMAKE_CUDA_ARCHITECTURES=75 ..

In [ ]:
# 4. [Каждый раз после git push] Подтянуть код, пересобрать, запустить
%cd /content/GPGPUTasks2026
!git fetch -q origin && git reset -q --hard origin/$BRANCH && git log --oneline -1
%cd /content/GPGPUTasks2026/build
!make -j$(nproc) 2>&1 | tail -20

print("\n\033[94m=== main_matrix_transpose ===\033[0m")
!./main_matrix_transpose 0
# умножение пока не реализовано, а его CPU-эталон идёт ~80 сек - включить, когда дойдём до части 2
# print("\n\033[94m=== main_matrix_multiply ===\033[0m")
# !./main_matrix_multiply 0

In [ ]:
# 5. По окончании — освободить GPU (экономит лимит)
from google.colab import runtime
runtime.unassign()
